# 🎬 Snoopy Clipper — Test Drive di Google Colab

Uji **seluruh pipeline** end-to-end tanpa menyentuh PC kamu: clone repo → install → transkripsi → otak Gemini → face tracking → smart placement → subtitle MrBeast-style → render 9:16 → preview hasil.

**Yang perlu disiapkan:** API key Gemini (gratis → https://aistudio.google.com/apikey). Itu saja.

Jalankan semua cell berurutan (Ctrl+F9 / Runtime → Run all).

In [ ]:
# 1. SETUP — clone repo + install dependensi (±2-3 menit, sekali per sesi)
!rm -rf snoopy-clipper
!git clone -q https://github.com/jorsbanana-nexux/snoopy-clipper.git
%cd snoopy-clipper
# Colab sudah punya: python, ffmpeg, opencv, numpy. Kita tambah sisanya:
!pip install -q yt-dlp faster-whisper google-genai
!ffmpeg -version | head -1
import cv2, numpy, faster_whisper, google.genai, yt_dlp
print("\n✅ SEMUA DEPENDENSI SIAP")

In [ ]:
# 2. API KEY GEMINI (otak AI) — paste di sini saat diminta, lalu enter.
#    Gratis di https://aistudio.google.com/apikey
import os
try:
    from google.colab import userdata  # pakai Secrets Colab kalau sudah disimpan
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ key diambil dari Secrets Colab")
except Exception:
    if "GEMINI_API_KEY" not in os.environ or not os.environ["GEMINI_API_KEY"]:
        key = input("Tempel API key Gemini kamu: ").strip()
        assert key, "API key wajib diisi"
        os.environ["GEMINI_API_KEY"] = key
        print("✅ key tersimpan untuk sesi ini")

In [ ]:
# 3. PILIH VIDEO UJI
#    MODE "URL"     : tempel URL YouTube/TikTok/IG apa pun.
#    MODE "UPLOAD"  : upload file mp4 dari PC/HP kamu (100% kebal blokir YouTube).
MODE = "URL"
VIDEO_URL = "https://www.youtube.com/watch?v=pf9vd2sny0M"  # contoh: 7 mnt, wajah+bicara
LOCAL_PATH = None
if MODE == "UPLOAD":
    from google.colab import files
    up = files.upload()
    assert up, "belum ada file diupload"
    LOCAL_PATH = "/content/snoopy-clipper/" + list(up.keys())[0]
    print("file:", LOCAL_PATH)

MAX_CLIPS = 3
os.environ["MAX_CLIPS"] = str(MAX_CLIPS)
print("MODE:", MODE, "|", LOCAL_PATH or VIDEO_URL)

In [ ]:
# 4. JALANKAN PIPELINE — sama persis dengan yang jalan di server lokal,
#    dengan progress + ETA realtime. Durasi tergantung panjang video
#    (transkripsi + render paling lama; lihat ETA-nya).
import sys, time, json
sys.path.insert(0, ".")
from backend import pipeline

job_id = pipeline.create_job(VIDEO_URL, local_path=LOCAL_PATH)
print("Job:", job_id)
last = None
job = {}
while True:
    job = pipeline.get_job(job_id)
    if job["status"] == "error":
        print("\n❌ ERROR:", job["error"])
        raise RuntimeError(job["error"])
    key = (job["step"], job["message"], job["pct"])
    if key != last:
        eta = job.get("eta_seconds")
        print(f"{job['pct']:3d}% | {job['step']:10s} | {job['message']}"
              + (f" | sisa ~{int(eta)} dtk" if eta else ""), flush=True)
        last = key
    if job["status"] == "done":
        break
    time.sleep(2)
print("\n🎉 SELESAI!")

In [ ]:
# 5. PREVIEW HASIL — putar klip langsung di notebook + unduh semua sebagai ZIP
from IPython.display import HTML, display
from pathlib import Path

for clip in job["clips"]:
    p = Path("library") / job["video_id"] / f"{clip['id']}.mp4"
    print(f"\n▶ {clip['title']}  |  skor {clip['score']}  |  {clip['duration']} dtk  |  {clip['width']}x{clip['height']}  |  {p.stat().st_size//1024//1024} MB")
    display(HTML(f'''<video width=270 controls src="{str(p).replace(' ', '%20')}" style="border-radius:12px"></video>'''))

!zip -q -r snoopy_clips.zip library/{job["video_id"]}
print("\n📦 Unduh semua klip: klik folder kiri Colab → snoopy-clipper/snoopy_clips.zip")

## Checklist yang baru saja diuji
- ✅ Download multi-platform (yt-dlp) + cache
- ✅ Transkripsi faster-whisper (timestamp per kata, anti-typo)
- ✅ Otak Gemini multimodal (transkrip + cuplikan frame)
- ✅ Face tracking v2 (fokus pembicara, look-ahead, pan mulus)
- ✅ Smart Placement (wajah, UI platform, teks bawaan, saliency)
- ✅ Subtitle Komika Axis (reveal per kata, bounce, emas penekanan)
- ✅ Grade GAME ULTRA + motion blur + render 1080x1920 9:16

Kalau semua hijau di sini, PC kamu tinggal: `pip install -r requirements.txt` → isi `.env` → `uvicorn backend.main:app` → buka localhost:8000.